# 03 — Modèle de référence : prédire les ventes

Ce notebook entraîne une première régression linéaire pour prédire `Sales`.

Le notebook précédent a déjà créé un découpage aléatoire reproductible de 80 % pour l'entraînement et 20 % pour le test. Nous réutilisons ces deux fichiers afin de garder exactement le même découpage et de ne jamais ajuster le prétraitement sur les données de test. Les fichiers sources ne sont pas modifiés.

## 1. Charger les jeux préparés

Les chemins sont relatifs à ce notebook, situé dans `notebooks/`. Chaque fichier contient les variables encodées et la cible `Sales`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

DATA_DIR = Path("../data")
TRAIN_FILE = DATA_DIR / "engineered_train.csv"
TEST_FILE = DATA_DIR / "engineered_test.csv"
TARGET = "Sales"

train_data = pd.read_csv(TRAIN_FILE)
test_data = pd.read_csv(TEST_FILE)

for split_name, split_data in (("entraînement", train_data), ("test", test_data)):
    if TARGET not in split_data.columns:
        raise ValueError(f"La cible {TARGET!r} est absente du jeu de {split_name}.")

if list(train_data.columns) != list(test_data.columns):
    raise ValueError("Les colonnes des jeux d'entraînement et de test ne correspondent pas.")

X_train = train_data.drop(columns=TARGET)
y_train = train_data[TARGET]
X_test = test_data.drop(columns=TARGET)
y_test = test_data[TARGET]

if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in X_train.dtypes):
    raise TypeError("Toutes les variables explicatives doivent être numériques.")
if not pd.api.types.is_numeric_dtype(y_train.dtype) or not pd.api.types.is_numeric_dtype(y_test.dtype):
    raise TypeError("La cible Sales doit être numérique pour une régression.")
if X_train.isna().any().any() or X_test.isna().any().any() or y_train.isna().any() or y_test.isna().any():
    raise ValueError("Des valeurs manquantes ont été détectées dans les données.")
if not np.isfinite(X_train.to_numpy()).all() or not np.isfinite(X_test.to_numpy()).all():
    raise ValueError("Des valeurs infinies ont été détectées dans les variables explicatives.")
if not np.isfinite(y_train.to_numpy()).all() or not np.isfinite(y_test.to_numpy()).all():
    raise ValueError("Des valeurs infinies ont été détectées dans la cible.")

print(f"Entraînement : {X_train.shape[0]} lignes, {X_train.shape[1]} variables")
print(f"Test : {X_test.shape[0]} lignes, {X_test.shape[1]} variables")
print(f"Cible : {TARGET}")

Entraînement : 7840 lignes, 1197 variables
Test : 1960 lignes, 1197 variables
Cible : Sales


## 2. Entraîner la régression linéaire

`Sales` est une valeur numérique : il s'agit donc d'un problème de **régression**. Le modèle est ajusté uniquement sur `X_train` et `y_train`, puis évalué sur les observations jamais vues du jeu de test.

In [2]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

model_mse = mean_squared_error(y_test, y_pred)
model_r2 = r2_score(y_test, y_pred)

print("Régression linéaire — résultats sur le jeu de test")
print(f"MSE : {model_mse:,.2f}")
print(f"R² : {model_r2:.4f}")

Régression linéaire — résultats sur le jeu de test
MSE : 558,897.44
R² : 0.1638


## 3. Comparer à une prédiction naïve

Pour mesurer l'apport du modèle, on le compare à une stratégie simple qui prédit toujours la moyenne des ventes du jeu d'entraînement. Une MSE plus faible et un R² plus élevé indiquent une meilleure performance. Le R² peut être négatif si le modèle fait moins bien que la moyenne sur le jeu de test.

In [3]:
baseline_prediction = np.full(len(y_test), y_train.mean(), dtype=float)
baseline_mse = mean_squared_error(y_test, baseline_prediction)
baseline_r2 = r2_score(y_test, baseline_prediction)

print("Référence naïve — prédire la moyenne du jeu d'entraînement")
print(f"MSE : {baseline_mse:,.2f}")
print(f"R² : {baseline_r2:.4f}")

if model_mse < baseline_mse and model_r2 > baseline_r2:
    print("Bilan : la régression linéaire fait mieux que la référence naïve sur les deux métriques.")
else:
    print("Bilan : la régression linéaire ne dépasse pas la référence naïve sur les deux métriques ; ce baseline doit être amélioré.")

Référence naïve — prédire la moyenne du jeu d'entraînement
MSE : 670,737.97
R² : -0.0035
Bilan : la régression linéaire fait mieux que la référence naïve sur les deux métriques.


## Interprétation

Le verdict imprimé ci-dessus compare les résultats sur les mêmes lignes de test. Si la régression réduit la MSE et augmente le R² par rapport à la prédiction constante de la moyenne, les variables préparées contiennent un signal prédictif utile. Sinon, cette première approche n'apporte pas encore de gain fiable par rapport à une simple estimation moyenne. Dans les deux cas, il s'agit d'un point de départ à conserver avant de tester des modèles plus élaborés.